# AI Image Lab — ComfyUI en Google Colab

Notebook para ejecutar ComfyUI con GPU de Colab, persistir modelos/LoRAs/workflows en Google Drive y acceder a la interfaz desde el navegador.

**Arquitectura:** Colab GPU → ComfyUI → Google Drive (`AI-Image`) → modelos / LoRAs / outputs.


## 1. Configuración
En Colab selecciona **Runtime → Change runtime type → GPU** antes de ejecutar las celdas.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

print('Entorno preparado.')


In [ ]:
!nvidia-smi

import torch
print('\nPyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GB')


## 2. Google Drive
Los modelos, LoRAs, workflows y resultados permanecerán en Drive aunque la sesión de Colab termine.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/AI-Image')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

for folder in [
    'models/checkpoints',
    'models/loras',
    'models/vae',
    'models/controlnet',
    'models/upscale_models',
    'models/clip',
    'output',
    'input',
    'workflows',
]:
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

print('Persistencia:', DRIVE_ROOT)


## 3. Instalar ComfyUI
Usamos el repositorio oficial y dejamos el código de ComfyUI en `/content` para no hacer que cada operación de la interfaz dependa de la velocidad de Drive. Los modelos y resultados sí viven en Drive.

In [ ]:
COMFY_DIR = Path('/content/ComfyUI')

if not (COMFY_DIR / 'main.py').exists():
    !git clone https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI

%cd /content/ComfyUI
!pip install -q -r requirements.txt

print('ComfyUI instalado en', COMFY_DIR)


## 4. Conectar modelos y resultados con Google Drive
Creamos enlaces simbólicos para que ComfyUI vea directamente las carpetas persistentes.

In [ ]:
import shutil

MODEL_DIR = COMFY_DIR / 'models'
DRIVE_MODELS = DRIVE_ROOT / 'models'

for name in ['checkpoints','loras','vae','controlnet','upscale_models','clip']:
    target = MODEL_DIR / name
    source = DRIVE_MODELS / name
    source.mkdir(parents=True, exist_ok=True)
    if target.is_symlink() or target.exists():
        if target.is_symlink():
            target.unlink()
        elif target.is_dir():
            shutil.rmtree(target)
    target.symlink_to(source, target_is_directory=True)

output_dir = COMFY_DIR / 'output'
if output_dir.is_symlink():
    output_dir.unlink()
elif output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.symlink_to(DRIVE_ROOT / 'output', target_is_directory=True)

print('Modelos:', DRIVE_MODELS)
print('Outputs:', DRIVE_ROOT / 'output')


## 5. ComfyUI Manager
El Manager permite instalar y administrar custom nodes desde la interfaz.

In [ ]:
MANAGER = COMFY_DIR / 'custom_nodes' / 'ComfyUI-Manager'
if not MANAGER.exists():
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
print('Manager listo.')


## 6. Descarga segura de modelos y LoRAs

Las descargas se hacen **directamente al disco** usando `wget`, no cargando el archivo completo en RAM. Esto es importante para checkpoints grandes como Flux en Colab.

Puedes usar URLs directas de archivos `.safetensors` de Hugging Face u otras fuentes compatibles. Si una descarga se corta, `wget -c` puede continuarla.


In [ ]:
MODEL_URLS = {
    'checkpoints': [],
    'loras': [],
    'vae': [],
    'controlnet': [],
}

print('Añade URLs a MODEL_URLS cuando quieras automatizar descargas.')


In [ ]:
import subprocess, urllib.parse

MODEL_URLS = {
    'checkpoints': [],
    'loras': [],
    'vae': [],
    'controlnet': [],
}

def filename_from_url(url):
    path = urllib.parse.urlparse(url).path
    filename = path.rstrip('/').split('/')[-1]
    if not filename:
        raise ValueError(f'No se pudo determinar el nombre del archivo: {url}')
    return filename

def is_safetensors_header_valid(path):
    """Comprueba solo el encabezado de safetensors sin cargar el modelo en RAM."""
    import struct
    try:
        with open(path, 'rb') as f:
            raw = f.read(8)
            if len(raw) != 8:
                return False
            header_len = struct.unpack('<Q', raw)[0]
            if header_len <= 0 or header_len > 100 * 1024 * 1024:
                return False
            header = f.read(header_len)
            return len(header) == header_len
    except Exception:
        return False

def download_file(url, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    filename = destination.name
    print(f'\nDescargando: {filename}')
    print('Destino:', destination)
    print('La descarga usa streaming al disco; no se carga el modelo completo en RAM.')

    # -c = continuar una descarga parcial; --progress=bar:force muestra progreso.
    cmd = [
        'wget', '-c', '--progress=bar:force', '--show-progress',
        '--tries=5', '--timeout=30', '--waitretry=5',
        '-O', str(destination), url,
    ]
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f'La descarga falló con código {result.returncode}: {filename}')

    size_gb = destination.stat().st_size / 1024**3
    print(f'Archivo descargado: {size_gb:.2f} GB')

    if destination.suffix.lower() == '.safetensors' and not is_safetensors_header_valid(destination):
        raise RuntimeError(
            f'El archivo {filename} parece incompleto/corrupto. '
            'Bórralo o vuelve a ejecutar la celda para reanudar la descarga.'
        )
    print('OK:', filename)

def download_urls(category):
    urls = MODEL_URLS.get(category, [])
    destination_dir = DRIVE_MODELS / category
    destination_dir.mkdir(parents=True, exist_ok=True)

    for url in urls:
        filename = filename_from_url(url)
        path = destination_dir / filename

        # No confiar únicamente en exists(): un archivo puede haber quedado incompleto
        # por una sesión de Colab agotada.
        if path.exists() and path.stat().st_size > 0:
            if path.suffix.lower() == '.safetensors':
                if is_safetensors_header_valid(path):
                    print('Ya existe y el encabezado es válido:', path.name)
                    continue
                print('Archivo safetensors incompleto/corrupto; wget intentará continuar:', path.name)
            else:
                print('Ya existe:', path.name)
                continue

        download_file(url, path)

for category in MODEL_URLS:
    download_urls(category)


## 7. Lanzar ComfyUI
La interfaz escucha en el puerto `8188`.

In [ ]:
%cd /content/ComfyUI

import subprocess, threading, time

if 'comfy_process' in globals() and comfy_process.poll() is None:
    print('ComfyUI ya está ejecutándose.')
else:
    comfy_process = subprocess.Popen(
        [sys.executable, 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    def stream_logs():
        for line in comfy_process.stdout:
            print(line, end='')

    threading.Thread(target=stream_logs, daemon=True).start()
    time.sleep(5)
    print('ComfyUI iniciado.')


## 8. URL pública con Cloudflare Tunnel
La URL es temporal y cambia al reiniciar la sesión.

In [ ]:
!wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared.deb >/dev/null 2>&1 || true

import subprocess, re, time
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

url = None
for _ in range(40):
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.25)
        continue
    print(line, end='')
    match = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

print('\nURL de ComfyUI:', url or 'No detectada; revisa los logs.')


## 9. Estructura final
```text
Google Drive/MyDrive/AI-Image/
├── models/
│   ├── checkpoints/
│   ├── loras/
│   ├── vae/
│   ├── controlnet/
│   └── upscale_models/
├── input/
├── output/
└── workflows/
```

### Próximos módulos
- Selector de modelos desde una celda de configuración.
- Descarga segura de checkpoints/LoRAs desde fuentes concretas.
- Workflows preconfigurados para SD 1.5 y SDXL.
- Panel sencillo para prompt, seed, steps, CFG y LoRA weights.
- Detección automática de T4/L4/A100 y ajuste de memoria.
